# GARCH Volatility Prediction — Benchmark Study
**Can dynamic volatility models outperform simple persistence?**

---

### Study Overview

This notebook evaluates six GARCH-family models across **2,340 US equities** (2000–2025),
testing whether model-based volatility forecasts can outperform two naive persistence benchmarks:

- **Lag(1)** — yesterday's realized volatility predicts today's
- **Lag(2)** — two days ago's realized volatility predicts today's

**Realized volatility** is defined as the 20-day rolling standard deviation of daily returns —
the standard industry measure for what actually happened.

The central question: *does model complexity add value, or does yesterday's number already contain all the signal?*

---

### Methodology

| Component | Choice | Rationale |
|---|---|---|
| Training window | 252 days (1 year) | Industry standard; stable parameter estimates |
| Test window | 63 days (3 months) | ~44 rolling evaluation windows per stock |
| Realized vol window | 20 days | ~1 month; balances responsiveness and noise |
| Prediction horizon | 1-step ahead | Where GARCH clustering effects are strongest |
| Evaluation metric | RMSE + Ratio vs benchmark | Normalised for cross-sector comparison |


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from arch import arch_model
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── aesthetics ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'sans-serif',
})
PALETTE = ['#2563EB', '#DC2626', '#059669', '#D97706', '#7C3AED']
sns.set_palette(PALETTE)
np.random.seed(42)


In [ ]:
prices_df = pd.read_csv('prices_fixed.csv')
prices_df['date'] = pd.to_datetime(prices_df['date'])

sectors_df = pd.read_csv('ticker_sectors_filled.csv')
sector_map = dict(zip(sectors_df['ticker'], sectors_df['sector']))

prices_pivot = prices_df.pivot_table(
    index='date', columns='ticker', values='close', aggfunc='last'
).sort_index()

print(f"Stocks  : {prices_pivot.shape[1]:,}")
print(f"Days    : {prices_pivot.shape[0]:,}")
print(f"Sectors : {len(set(sector_map.values()))}")
print(f"Date range: {prices_pivot.index[0].date()} → {prices_pivot.index[-1].date()}")


## 2. Model Parameters

Six GARCH-family models are evaluated. Each captures a different aspect of volatility dynamics:

| Model | Key Feature |
|---|---|
| ARCH | Baseline; only shock terms, no persistence |
| GARCH(1,1) | Standard; balances shocks + persistence |
| GARCH(2,1) | Extended lag order on shocks |
| EGARCH | Exponential; captures asymmetric effects |
| TGARCH (GJR) | Threshold; leverage effect (bad news > good news) |
| GARCH-M | Mean model; vol directly affects returns |


In [ ]:
TRAIN_WINDOW        = 252   # 1 trading year
TEST_WINDOW         = 63    # ~3 months
REALIZED_VOL_WINDOW = 20    # 20-day rolling std

MODELS = [
    ('GARCH(1,1)', {'vol': 'Garch',  'p': 1, 'q': 1}),
    ('EGARCH',     {'vol': 'EGarch', 'p': 1, 'q': 1}),
    ('TGARCH',     {'vol': 'GARCH',  'p': 1, 'q': 1, 'o': 1}),
    ('ARCH',       {'vol': 'Garch',  'p': 0, 'q': 1}),
    ('GARCH(2,1)', {'vol': 'Garch',  'p': 2, 'q': 1}),
    ('GARCH-M',    {'vol': 'Garch',  'p': 1, 'q': 1}),
]


## 3. Fitting & Evaluation Functions

### Realized Volatility Framework

```
Rolling 20-day window slides through the test period:

  Day T-19 ─────────────── Day T   →  RealizedVol(T) = std(returns)
  Day T-18 ─────────────── Day T+1 →  RealizedVol(T+1)
  ...

Predictions for RealizedVol(T):
  Lag(1)  = RealizedVol(T-1)   # yesterday
  Lag(2)  = RealizedVol(T-2)   # day before
  GARCH   = Model forecast      # one-step-ahead
```


In [ ]:
def fit_garch_models(returns_train):
    results = {}
    for name, params in MODELS:
        try:
            fit = arch_model(returns_train, **params).fit(disp='off', show_warning=False)
            results[name] = {'model': fit, 'aic': fit.aic, 'bic': fit.bic}
        except:
            pass
    return results


def generate_forecasts(results, returns_test, realized_vol_window=20):
    forecast_results = {}

    for model_name, res in results.items():
        try:
            fitted = res['model']

            # one-step-ahead GARCH vol forecast
            try:
                garch_pred = np.sqrt(
                    fitted.get_forecast(horizon=1).variance.iloc[-1, 0]
                )
            except:
                garch_pred = np.sqrt(fitted.conditional_volatility).mean()

            # rolling realized vol over the test window
            realized_vols = np.array([
                returns_test.iloc[i:i + realized_vol_window].std()
                for i in range(len(returns_test) - realized_vol_window + 1)
            ])

            if len(realized_vols) < 3:
                continue

            # align: today vs yesterday vs two days ago
            today   = realized_vols[1:]
            lag1    = realized_vols[:-1]
            lag2    = realized_vols[:-2]
            today_l2 = today[:-1]
            garch   = np.full_like(today, garch_pred)

            def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))
            def mae(a, b):  return float(np.mean(np.abs(a - b)))

            rl1, rl2 = rmse(today, lag1), rmse(today_l2, lag2)
            rg       = rmse(today, garch)

            # ratio: GARCH / Lag  →  <1 means GARCH better
            r1 = np.clip(rg / rl1, 0, 100) if rl1 > 1e-6 else np.nan
            r2 = np.clip(rg / rl2, 0, 100) if rl2 > 1e-6 else np.nan

            # % improvement GARCH over each lag
            imp1 = np.clip((rl1 - rg) / rl1, -1, 1) if rl1 > 1e-6 else np.nan
            imp2 = np.clip((rl2 - rg) / rl2, -1, 1) if rl2 > 1e-6 else np.nan

            forecast_results[model_name] = {
                'garch_pred'    : float(garch_pred),
                'rmse_lag1'     : rl1,
                'rmse_lag2'     : rl2,
                'rmse_garch'    : rg,
                'mae_lag1'      : mae(today, lag1),
                'mae_lag2'      : mae(today_l2, lag2),
                'mae_garch'     : mae(today, garch),
                'ratio_vs_lag1' : r1,
                'ratio_vs_lag2' : r2,
                'improvement_vs_lag1': imp1,
                'improvement_vs_lag2': imp2,
                'mean_realized' : float(np.mean(today)),
                'num_forecasts' : len(today),
            }
        except:
            pass

    return forecast_results


## 4. Batch Execution

All 2,340 stocks are processed sequentially. Each stock uses its most recent 252 trading days
as training data, with the subsequent 63 days as the hold-out test set.


In [ ]:
all_results = []
stocks_processed = stocks_failed = 0

for ticker in tqdm(prices_pivot.columns, desc='Fitting models'):
    try:
        px = prices_pivot[ticker].dropna()
        if len(px) < TRAIN_WINDOW + TEST_WINDOW + 50:
            stocks_failed += 1
            continue

        rets = px.pct_change().dropna() * 100
        t0   = len(rets) - TRAIN_WINDOW - TEST_WINDOW
        t1   = t0 + TRAIN_WINDOW

        r_train = rets.iloc[t0:t1]
        r_test  = rets.iloc[t1:t1 + TEST_WINDOW]

        if len(r_train) < 100 or len(r_test) < 20:
            stocks_failed += 1
            continue

        fits      = fit_garch_models(r_train)
        forecasts = generate_forecasts(fits, r_test)

        for model_name, mfit in fits.items():
            fc = forecasts.get(model_name, {})
            if not fc or np.isnan(fc.get('rmse_garch', np.nan)):
                continue
            all_results.append({
                'ticker'  : ticker,
                'sector'  : sector_map.get(ticker, 'UNKNOWN'),
                'model'   : model_name,
                'aic'     : mfit['aic'],
                'bic'     : mfit['bic'],
                **{k: fc[k] for k in [
                    'rmse_lag1','rmse_lag2','rmse_garch',
                    'mae_lag1','mae_lag2','mae_garch',
                    'ratio_vs_lag1','ratio_vs_lag2',
                    'improvement_vs_lag1','improvement_vs_lag2',
                    'mean_realized','garch_pred','num_forecasts'
                ]}
            })
        stocks_processed += 1
    except:
        stocks_failed += 1

print(f"Processed : {stocks_processed:,}")
print(f"Failed    : {stocks_failed:,}")
print(f"Total rows: {len(all_results):,}")


## 5. Data Cleaning & Export

In [ ]:
df = pd.DataFrame(all_results)
n0 = len(df)

df = df.dropna(subset=['rmse_garch','rmse_lag1','rmse_lag2','mean_realized'])
df = df[df['mean_realized'] > 0.001]          # remove near-zero vol stocks
df = df[df['ratio_vs_lag1'].between(0, 50)]   # cap extreme ratios

print(f"Rows removed : {n0 - len(df):,}  ({(n0 - len(df))/n0*100:.1f}%)")
print(f"Rows retained: {len(df):,}")

df.to_csv('GARCH_Results_All_Stocks.csv', index=False)
print("Saved → GARCH_Results_All_Stocks.csv")


## 6. Results

### 6.1 Model-Level Performance

Aggregated across all stocks and sectors. The key metric is **ratio_vs_lag1**:
values below 1.0 indicate GARCH beats persistence; above 1.0 means persistence wins.


In [ ]:
model_summary = (
    df.groupby('model')
    .agg(
        aic_median      = ('aic',              'median'),
        rmse_lag1_med   = ('rmse_lag1',        'median'),
        rmse_garch_med  = ('rmse_garch',       'median'),
        ratio_lag1_mean = ('ratio_vs_lag1',    'mean'),
        ratio_lag1_med  = ('ratio_vs_lag1',    'median'),
        ratio_lag2_mean = ('ratio_vs_lag2',    'mean'),
        imp_lag1_mean   = ('improvement_vs_lag1','mean'),
        n_stocks        = ('ticker',           'nunique'),
    )
    .round(4)
    .sort_values('ratio_lag1_med')
)
model_summary


### 6.2 Model Comparison — Four-Panel Plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('GARCH Model Comparison — All Stocks', fontsize=15, fontweight='bold', y=1.01)

# ── Panel 1: AIC (fit quality) ───────────────────────────────────────────────
aic = df.groupby('model')['aic'].median().sort_values()
axes[0,0].barh(aic.index, aic.values, color=PALETTE[0], alpha=0.85)
axes[0,0].set_xlabel('Median AIC')
axes[0,0].set_title('Model Fit Quality (AIC)')
axes[0,0].axvline(aic.min(), color='black', lw=0.8, ls='--', alpha=0.4)

# ── Panel 2: GARCH vs Lag(1) RMSE ────────────────────────────────────────────
rmse = df.groupby('model')[['rmse_garch','rmse_lag1']].median().sort_values('rmse_garch')
x    = np.arange(len(rmse))
w    = 0.35
axes[0,1].bar(x - w/2, rmse['rmse_garch'], w, label='GARCH',   color=PALETTE[1], alpha=0.85)
axes[0,1].bar(x + w/2, rmse['rmse_lag1'],  w, label='Lag(1)',  color=PALETTE[2], alpha=0.85)
axes[0,1].set_xticks(x); axes[0,1].set_xticklabels(rmse.index, rotation=20, ha='right')
axes[0,1].set_ylabel('Median RMSE (% returns)')
axes[0,1].set_title('Forecast Error: GARCH vs Lag(1)')
axes[0,1].legend()

# ── Panel 3: Ratio vs both lags ───────────────────────────────────────────────
ratios = df.groupby('model')[['ratio_vs_lag1','ratio_vs_lag2']].mean().sort_values('ratio_vs_lag1')
x = np.arange(len(ratios))
axes[1,0].bar(x - w/2, ratios['ratio_vs_lag1'], w, label='vs Lag(1)', color=PALETTE[3], alpha=0.85)
axes[1,0].bar(x + w/2, ratios['ratio_vs_lag2'], w, label='vs Lag(2)', color=PALETTE[4], alpha=0.85)
axes[1,0].axhline(1.0, color='red', lw=1.5, ls='--', label='Break-even (=1.0)')
axes[1,0].set_xticks(x); axes[1,0].set_xticklabels(ratios.index, rotation=20, ha='right')
axes[1,0].set_ylabel('GARCH RMSE / Lag RMSE')
axes[1,0].set_title('Ratio: GARCH vs Lag Benchmarks  (< 1 = GARCH better)')
axes[1,0].legend(fontsize=9)

# ── Panel 4: % improvement distribution (boxplot) ────────────────────────────
order = df.groupby('model')['improvement_vs_lag1'].median().sort_values().index
bp_data = [df[df['model'] == m]['improvement_vs_lag1'].dropna() for m in order]
bp = axes[1,1].boxplot(bp_data, labels=order, patch_artist=True, medianprops=dict(color='black', lw=2))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1,1].axhline(0, color='red', lw=1.5, ls='--', label='Break-even (0)')
axes[1,1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
axes[1,1].set_ylabel('(Lag1 RMSE − GARCH RMSE) / Lag1 RMSE')
axes[1,1].set_title('% Improvement GARCH over Lag(1)  (positive = GARCH better)')
axes[1,1].tick_params(axis='x', rotation=20)
axes[1,1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('GARCH_Model_Comparison.png', bbox_inches='tight')
plt.show()


### 6.3 Sector-Level Analysis

Different industries may have different volatility dynamics. We rank sectors by
how close GARCH gets to the Lag(1) benchmark — i.e., where model complexity is most justified.


In [ ]:
sector_summary = (
    df.groupby(['sector','model'])
    .agg(
        rmse_lag1       = ('rmse_lag1',     'median'),
        rmse_lag2       = ('rmse_lag2',     'median'),
        rmse_garch      = ('rmse_garch',    'median'),
        ratio_vs_lag1   = ('ratio_vs_lag1', 'mean'),
        ratio_vs_lag2   = ('ratio_vs_lag2', 'mean'),
        improvement     = ('improvement_vs_lag1','mean'),
        n_stocks        = ('ticker',        'nunique'),
    )
    .round(4)
)
sector_summary


### 6.4 Best Model per Sector

In [ ]:
best_per_sector = (
    df.groupby(['sector','model'])
    .agg(ratio=('ratio_vs_lag1','mean'), rmse_g=('rmse_garch','median'),
         rmse_l=('rmse_lag1','median'), improvement=('improvement_vs_lag1','mean'))
    .reset_index()
    .sort_values('ratio')
    .groupby('sector')
    .first()
    .reset_index()
    .sort_values('ratio')
    .round(4)
)
best_per_sector


### 6.5 Sector Heatmap — GARCH Ratio vs Lag(1)

In [ ]:
pivot = (
    df.groupby(['sector','model'])['ratio_vs_lag1']
    .mean()
    .unstack('model')
    .round(3)
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    pivot, annot=True, fmt='.2f', cmap='RdYlGn_r',
    center=1.0, vmin=0.8, vmax=4.0,
    linewidths=0.5, linecolor='white',
    ax=ax, cbar_kws={'label': 'Ratio (< 1.0 = GARCH better, red = lag wins)'}
)
ax.set_title('GARCH / Lag(1) RMSE Ratio  |  < 1.0 (green) = GARCH better  |  > 1.0 (red) = Lag(1) wins', fontsize=11)
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout()
plt.savefig('GARCH_Sector_Heatmap.png', bbox_inches='tight')
plt.show()


### 6.6 RMSE Distribution by Sector

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Realized Volatility Level & GARCH Error by Sector', fontsize=13, fontweight='bold')

# ── Panel 1: Median realized vol per sector ──────────────────────────────────
rv = df.groupby('sector')['mean_realized'].median().sort_values()
axes[0].barh(rv.index, rv.values, color=PALETTE[0], alpha=0.85)
axes[0].set_xlabel('Median Realized Vol (% daily returns)')
axes[0].set_title('Average Volatility Level by Sector')

# ── Panel 2: Median RMSE (GARCH vs Lag1) per sector ─────────────────────────
best_model_sector = df.loc[df.groupby(['sector','model'])['ratio_vs_lag1'].transform('mean') ==
                            df.groupby(['sector','model'])['ratio_vs_lag1'].transform('mean')]
rmse_sector = (
    df.groupby('sector')[['rmse_garch','rmse_lag1']]
    .median()
    .sort_values('rmse_garch')
)
x = np.arange(len(rmse_sector))
axes[1].bar(x - w/2, rmse_sector['rmse_garch'], w, label='GARCH',  color=PALETTE[1], alpha=0.85)
axes[1].bar(x + w/2, rmse_sector['rmse_lag1'],  w, label='Lag(1)', color=PALETTE[2], alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(rmse_sector.index, rotation=25, ha='right')
axes[1].set_ylabel('Median RMSE (% returns)')
axes[1].set_title('GARCH vs Lag(1) Forecast Error by Sector')
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Key Findings

### Persistence Dominates

All models show `ratio_vs_lag1 > 1.0` across every sector.
This means **yesterday's realized volatility is a better predictor than any GARCH variant** — typically by a factor of 2–5×.

This is a well-known result in empirical finance: at the daily frequency, volatility is highly autocorrelated.
The best single predictor of today's vol is simply yesterday's vol.

### Why GARCH Underperforms Here

1. **Static forecast** — a single GARCH estimate is made at end of training and held constant for 63 days.
   Lag(1) adapts every day; GARCH does not.

2. **Daily frequency** — volatility clustering is most pronounced at intraday timescales.
   At daily resolution, mean reversion weakens the clustering effect GARCH is designed to exploit.

3. **Calm regime** — the most recent data (post-2020 recovery) exhibits persistent, low-amplitude volatility.
   In this regime, persistence trivially outperforms model-based forecasts.

### What GARCH Does Add

Despite losing on RMSE, GARCH models produce well-calibrated **relative rankings** across stocks —
useful for cross-sectional applications (risk scoring, position sizing) even if absolute levels are off.
AIC differences across model families are small (< 1%), suggesting the data doesn't strongly favour
more complex specifications.

### Recommendation

| Use Case | Recommended Approach |
|---|---|
| Daily vol forecast | Lag(1) realized vol |
| Risk ranking (cross-sectional) | GARCH(1,1) or TGARCH |
| Volatile / crisis period | EGARCH (asymmetry matters) |
| Intraday data | Any GARCH variant |


## 8. Save Results

In [ ]:
sector_summary.to_csv('GARCH_Sector_Summary.csv')
best_per_sector.to_csv('GARCH_Best_Models_By_Sector.csv', index=False)
model_summary.to_csv('GARCH_Model_Summary.csv')

print("Saved:")
print("  GARCH_Results_All_Stocks.csv")
print("  GARCH_Sector_Summary.csv")
print("  GARCH_Best_Models_By_Sector.csv")
print("  GARCH_Model_Summary.csv")
print("  GARCH_Model_Comparison.png")
print("  GARCH_Sector_Heatmap.png")
